In [15]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

In [16]:
df = pd.read_csv("../Data/final_clean_data.csv")

selected_columns = [
    "MONTH","DAY_OF_MONTH","DAY_OF_WEEK",
    "OP_UNIQUE_CARRIER","ORIGIN","DEST",
    "CRS_DEP_TIME","CRS_ARR_TIME",
    "CRS_ELAPSED_TIME","DISTANCE",
    "HourlyDewPointTemperature","HourlyDryBulbTemperature",
    "HourlyRelativeHumidity","HourlyVisibility","HourlyWindSpeed",
    "DEP_DEL15"
]

df_classifi = df[selected_columns].copy()
print("Shape:", df_classifi.shape)

Shape: (47576, 16)


In [17]:
# Check target distribution
print(df_classifi["DEP_DEL15"].value_counts(normalize=True))

DEP_DEL15
0.0    0.786111
1.0    0.213889
Name: proportion, dtype: float64


#### Encode categorical variables

In [18]:
label_cols = ["OP_UNIQUE_CARRIER", "ORIGIN", "DEST"]
le = LabelEncoder()

for col in label_cols:
    df_classifi[col + "_ENC"] = le.fit_transform(df_classifi[col])

df_classifi = df_classifi.drop(columns=label_cols)
df_classifi.head()

,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,CRS_DEP_TIME,CRS_ARR_TIME,CRS_ELAPSED_TIME,DISTANCE,HourlyDewPointTemperature,HourlyDryBulbTemperature,HourlyRelativeHumidity,HourlyVisibility,HourlyWindSpeed,DEP_DEL15,OP_UNIQUE_CARRIER_ENC,ORIGIN_ENC,DEST_ENC
0,4,1,2,6.53,9.33,168.0,1020.0,33.0,36.0,89.0,5.0,10.0,0.0,0,1,62
1,4,1,2,15.45,18.12,160.0,1020.0,33.0,45.0,63.0,10.0,10.0,1.0,0,1,62
2,4,1,2,7.33,15.32,299.0,2279.0,37.0,43.0,80.0,10.0,6.0,0.0,0,4,14
3,4,1,2,13.83,21.83,300.0,2279.0,41.0,49.0,74.0,10.0,6.0,0.0,0,4,14
4,4,1,2,22.00,6.00,300.0,2279.0,41.0,45.0,86.0,10.0,7.0,0.0,0,4,14


#### Feature Engineering

In [19]:
# 1. Departure time group (CRS_DEP_TIME)
def time_of_day(hour):
    if 5 <= hour < 9:
        return "morning_peak"
    elif 9 <= hour < 15:
        return "midday"
    elif 15 <= hour < 19:
        return "evening_peak"
    else:
        return "night"

df_classifi["TIME_OF_DAY"] = df_classifi["CRS_DEP_TIME"].apply(time_of_day)

# Encode TIME_OF_DAY
le_time = LabelEncoder()
df_classifi["TIME_OF_DAY_ENC"] = le_time.fit_transform(df_classifi["TIME_OF_DAY"])

# 2. Create a temperature difference feature
df_classifi["TEMP_DIFF"] = (
    df_classifi["HourlyDryBulbTemperature"] - df_classifi["HourlyDewPointTemperature"]
)

df_classifi = df_classifi.drop(columns=["TIME_OF_DAY"])
df_classifi[["CRS_DEP_TIME", "TIME_OF_DAY_ENC", "TEMP_DIFF"]].head()

,CRS_DEP_TIME,TIME_OF_DAY_ENC,TEMP_DIFF
0,6.53,2,3.0
1,15.45,0,12.0
2,7.33,2,6.0
3,13.83,1,8.0
4,22.00,3,4.0


In [20]:
output_path = "../Data/final_classification_ready_v2.csv"
df_classifi.to_csv(output_path, index=False)